In [0]:
from pathlib import Path
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

from datetime import datetime, timedelta, timezone

import traceback

In [0]:
display(
    pd.read_excel("/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_touchpoint_loading_blob_config_20260714.xlsx")
)

In [0]:
display(
    pd.read_excel("/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_blob_config_20260714.xlsx")
)

In [0]:


display(
    pd.read_excel("/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_blob_config_20260714-CBR&Landing.xlsx")
)

In [0]:


'''
/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_touchpoint_loading_blob_config_20260714.xlsx

/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_blob_config_20260714.xlsx

/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_blob_config_20260714-CBR&Landing.xlsx
'''
# market in ['a', 'b']
market_condition = dbutils.widgets.get("Market_Conddition")
CONFIG_PATH = Path("/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_blob_config_20260714-CBR&Landing.xlsx")

print(f"market_condition: {market_condition}")
print(f"CONFIG_PATH: {CONFIG_PATH}")

# MDM Mysqsl连接信息
# MYSQL_HOST = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_HOST')
# MYSQL_USER = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_USER')
# MYSQL_PASSWORD = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_PSWD')
MYSQL_HOST = "mysqlflex-ap-southeastasia-prod-cepa-talend-02.mysql.database.azure.com"
MYSQL_USER = "talend_mdm_read_only@mysql-ap-southeastasia-prod-cepa-talend-02"
MYSQL_PASSWORD = "Q4&4c9JpMSb5A=xa"
MYSQL_DRIVER = "com.mysql.cj.jdbc.Driver"
MYSQL_USE_SSL = True
DEFAULT_ID_KEY = "id"
DEFAULT_PARTITION_SIZE = 200000
DEFAULT_FETCH_SIZE = 10000
DEFAULT_STRING_PARTITIONS = 16
MAX_CONCURRENT_TABLES = 2
TARGET_RECORDS_PER_FILE = 500000
MIN_OUTPUT_PARTITIONS = 1
MAX_OUTPUT_PARTITIONS = 200



# 定义 UTC+8 时区
TZ_UTC8 = timezone(timedelta(hours=8))

def print_log(message, level="INFO"):
    """打印带 UTC+8 时间戳的日志"""
    timestamp = datetime.now(TZ_UTC8).strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")




def build_jdbc_url(database: str) -> str:
    ssl_str = "&useSSL=true&enabledTLSProtocols=TLSv1.2" if MYSQL_USE_SSL else ""
    return (
        f"jdbc:mysql://{MYSQL_HOST}:3306/{database}"
        f"?serverTimezone=UTC&rewriteBatchedStatements=true&useUnicode=true&characterEncoding=UTF-8&zeroDateTimeBehavior=CONVERT_TO_NULL&useCompression=true{ssl_str}"
    )



def read_mysql_int_pk(
    market: str,
    database: str,
    table_name: str,
    id_key: str = DEFAULT_ID_KEY,
    partition_size: int = DEFAULT_PARTITION_SIZE,
    fetch_size: int = DEFAULT_FETCH_SIZE,
    trace_id: str = "",
):
    # 整型主键：先取主键最小/最大值，再按范围并行拉取。
    print_log(f"{trace_id} loading {market} table: {database}.{table_name}, pk: {id_key}, fun: read_mysql_int_pk")

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
    }

    edge_df = spark.read.jdbc(
        url=mysql_url,
        table=(
            f"(SELECT MIN({id_key}) as min_id, MAX({id_key}) as max_id "
            f"FROM {table_name}) as tmp"
        ),
        properties=mysql_properties,
    )
    edge_scope = edge_df.first()
    if edge_scope is None:
        df = (
                spark.read.format("jdbc")
                .option("url", mysql_url)
                .option("dbtable", table_name)
                .option("user", MYSQL_USER)
                .option("password", MYSQL_PASSWORD)
                .option("driver", MYSQL_DRIVER)
                .load()
            )

        return df, None, None

    lower_bound = edge_scope["min_id"]
    upper_bound = edge_scope["max_id"]
    if lower_bound is None or upper_bound is None:
        df = (
            spark.read.format("jdbc")
            .option("url", mysql_url)
            .option("dbtable", table_name)
            .option("user", MYSQL_USER)
            .option("password", MYSQL_PASSWORD)
            .option("driver", MYSQL_DRIVER)
            .load()
        )
        return df, None, None

    num_partitions = ((upper_bound - lower_bound) // partition_size) + 1
    print(
        f"{trace_id} lower_bound: {lower_bound}, upper_bound: {upper_bound}, "
        f"num_partitions: {num_partitions}"
    )

    df = (
        spark.read.format("jdbc")
        .option("url", mysql_url)
        .option("driver", MYSQL_DRIVER)
        .option("dbtable", table_name)
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("partitionColumn", id_key)
        .option("lowerBound", lower_bound)
        .option("upperBound", upper_bound)
        .option("numPartitions", num_partitions)
        .option("fetchSize", fetch_size)
        .load()
    )

    print_log(f"{trace_id} finished")
    return df, lower_bound, upper_bound


def read_mysql_string_pk(
    market: str,
    database: str,
    table_name: str,
    id_key: str = DEFAULT_ID_KEY,
    num_partitions: int = DEFAULT_STRING_PARTITIONS,
    fetch_size: int = DEFAULT_FETCH_SIZE,
    trace_id: str = "",
):
    # 字符串主键：用 CRC32 分桶生成 predicates，避免单线程全表扫描。
    print_log(f"{trace_id} loading {market} table: {database}.{table_name}, pk: {id_key}, fun: read_mysql_string_pk")

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
        "fetchsize": str(fetch_size),
    }
    predicates = [
        f"MOD(CRC32({id_key}), {num_partitions}) = {idx}"
        for idx in range(num_partitions)
    ]

    df = spark.read.jdbc(
        url=mysql_url,
        table=table_name,
        predicates=predicates,
        properties=mysql_properties,
    )

    print_log(f"{trace_id} finished")
    return df, None, None




def read_mysql_by_time_range(
    market: str,
    database: str,
    table_name: str,
    id_key: str,
    num_partitions: int = 200,
    fetch_size: int = -2147483648,
    trace_id: str = "",
):
    print_log(f"{trace_id} loading {market} table: {database}.{table_name}, pk: {id_key}, fun: read_mysql_by_time_range")

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
        "fetchsize": str(fetch_size),
        "useServerPrepStmts": "true"
    }

    # 1. 获取最小和最大时间（秒级精度）
    min_max_sql = f"SELECT MIN({id_key}) AS min_ts, MAX({id_key}) AS max_ts FROM {table_name}"
    min_max_df = spark.read.jdbc(mysql_url, f"({min_max_sql}) AS tmp", properties=mysql_properties)
    row = min_max_df.collect()[0]
    min_ts = row["min_ts"]
    max_ts = row["max_ts"]
    if min_ts is None or max_ts is None:
        return spark.read.jdbc(mysql_url, table_name, properties=mysql_properties)  # 表为空

    # 2. 计算每个分区的起止时间
    total_seconds = (max_ts - min_ts).total_seconds()
    step_seconds = total_seconds / num_partitions
    predicates = []
    start = min_ts
    for i in range(num_partitions):
        if i == num_partitions - 1:
            end = max_ts + timedelta(seconds=1)  # 确保包含最大值
        else:
            end = min_ts + timedelta(seconds=(i + 1) * step_seconds)
        # 构造 predicate，注意时间格式需转成 MySQL 接受的字符串
        predicates.append(
            f"{id_key} >= '{start.strftime('%Y-%m-%d %H:%M:%S')}' AND "
            f"{id_key} < '{end.strftime('%Y-%m-%d %H:%M:%S')}'"
        )
        start = end

    # print(predicates)

    # 3. 并行读取
    df = spark.read.jdbc(
        url=mysql_url,
        table=table_name,
        predicates=predicates,
        properties=mysql_properties
    )
    return df, min_ts, max_ts






def _build_uuid_range_predicates(
    id_key: str,
    prefix_len: int = 1,
    uuid_has_dash: bool = True,
):
    """
    生成 UUID 字符串范围分片条件。
    prefix_len=1 -> 16 分片；prefix_len=2 -> 256 分片（不建议默认直接256，先压测）。
    """
    hex_chars = "0123456789abcdef"
    predicates = []

    if prefix_len == 1:
        for i, c in enumerate(hex_chars):
            if i < len(hex_chars) - 1:
                next_c = hex_chars[i + 1]
                # 用 >= and < 做范围，通常可走 B-Tree 索引
                predicates.append(f"{id_key} >= '{c}' AND {id_key} < '{next_c}'")
            else:
                predicates.append(f"{id_key} >= '{c}'")
        return predicates

    if prefix_len == 2:
        prefixes = [a + b for a in hex_chars for b in hex_chars]  # 256
        for i, p in enumerate(prefixes):
            if i < len(prefixes) - 1:
                p_next = prefixes[i + 1]
                predicates.append(f"{id_key} >= '{p}' AND {id_key} < '{p_next}'")
            else:
                predicates.append(f"{id_key} >= '{p}'")
        return predicates

    raise ValueError("prefix_len only supports 1 or 2")


def read_mysql_uuid_pk(
    market: str,
    database: str,
    table_name: str,
    id_key: str = DEFAULT_ID_KEY,             # UUID 主键列
    num_partitions: int = DEFAULT_STRING_PARTITIONS,  # 这里不再直接使用；由 prefix_len 决定
    fetch_size: int = DEFAULT_FETCH_SIZE,
    trace_id: str = "",
    prefix_len: int = 2,                      # 1=16分片，2=256分片
):
    print_log(
        f"{trace_id} loading {market} table: {database}.{table_name}, "
        f"pk: {id_key}, fun: read_mysql_uuid_pk (uuid-range)"
    )

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
        "fetchsize": str(fetch_size),
    }

    predicates = _build_uuid_range_predicates(
        id_key=id_key,
        prefix_len=prefix_len,
    )

    df = spark.read.jdbc(
        url=mysql_url,
        table=table_name,
        predicates=predicates,
        properties=mysql_properties,
    )

    # print_log(f"{trace_id} finished, partitions={len(predicates)}")
    return df, None, None






def clamp_partitions(value: int) -> int:
    return max(MIN_OUTPUT_PARTITIONS, min(MAX_OUTPUT_PARTITIONS, value))


def pick_output_partitions(row_estimate: int | None) -> int:
    # 根据估算行数控制输出分区，平衡写入并行度与小文件数量。
    if not row_estimate or row_estimate <= 0:
        return clamp_partitions(DEFAULT_STRING_PARTITIONS)
    target = (row_estimate // TARGET_RECORDS_PER_FILE) + 1
    return clamp_partitions(target)

def load_config(condition_str) -> pd.DataFrame:
    # condition_str:   market in ['a', 'b']

    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Config not found at {CONFIG_PATH}. Update CONFIG_PATH if needed."
        )
    return pd.read_excel(CONFIG_PATH).replace({np.nan: None}).query(condition_str)




display(load_config(market_condition))

In [0]:
def is_integer_pk(pk_type: str) -> bool:
    pk_type_lower = pk_type.strip().lower()
    return pk_type_lower.startswith(
        ("int", "bigint", "smallint", "tinyint", "mediumint")
    )


def is_datetime_pk(pk_type: str) -> bool:
    pk_type_lower = pk_type.strip().lower()
    return pk_type_lower.startswith(
        ("datetime")
    )

def is_uuid_pk(pk_type: str) -> bool:
    pk_type_lower = pk_type.strip().lower()
    return pk_type_lower.startswith(
        ("uuid")
    ) 

def process_table(item: dict) -> None:
    market = str(item.get("market", "")).strip()
    database = str(item.get("source_database", "")).strip()
    table = str(item.get("source_table", "")).strip()
    target_path = str(item.get("target_path", "")).strip()
    primary_key = item.get("primary_key")
    pk_type = item.get("pk_type")

    if not (market and database and table and target_path):
        return

    trace_id = f"[{market}-{database}-{table}]"

    try:
        # print_log(f"primary_key: {primary_key}, {type(primary_key)}")

        if primary_key == None:
            print_log(f"{trace_id} loading {market} table: {database}.{table}, pk: (none)")
            jdbc_url = build_jdbc_url(database)
            df = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("dbtable", table)
                .option("user", MYSQL_USER)
                .option("password", MYSQL_PASSWORD)
                .option("driver", MYSQL_DRIVER)
                .load()
            )
            lower_bound = None
            upper_bound = None
        elif is_integer_pk(pk_type):
            df, lower_bound, upper_bound = read_mysql_int_pk(
                market,
                database,
                table,
                primary_key,
                trace_id=trace_id,
            )
        elif is_datetime_pk(pk_type):
            df, lower_bound, upper_bound = read_mysql_by_time_range(
                market,
                database,
                table,
                primary_key,
                trace_id=trace_id,
            )
        elif is_uuid_pk(pk_type):
            df, lower_bound, upper_bound = read_mysql_uuid_pk(
                market,
                database,
                table,
                primary_key,
                fetch_size = 10,
                trace_id=trace_id,
                prefix_len = 2

            )
        else:
            df, lower_bound, upper_bound = read_mysql_string_pk(
                market,
                database,
                table,
                primary_key,
                trace_id=trace_id,
            )

        if df is None:
            print_log(f"{trace_id} [SKIP] market={market} database={database} table={table}")
            return

        # 整型主键场景可用主键范围粗略估算行数，用于写入前分区调整。
        # row_estimate = None
        # if lower_bound is not None and upper_bound is not None:
        #     row_estimate = max(0, int(upper_bound) - int(lower_bound) + 1)

        # 写 Delta 前先调分区，减少过多小文件或单任务过重。
        # target_partitions = pick_output_partitions(row_estimate)
        # current_partitions = df.rdd.getNumPartitions()
        # if target_partitions < current_partitions:
        #     df = df.coalesce(target_partitions)
        # elif target_partitions > current_partitions:
        #     df = df.repartition(target_partitions)

        (
            df.write.format("delta")
            .option("maxRecordsPerFile", TARGET_RECORDS_PER_FILE)
            .mode("overwrite")
            .save(target_path)
        )

        print_log(f"{trace_id} [OK] -> {target_path}")
    except Exception as exc:
        print_log(f"{trace_id} [ERROR] {type(exc).__name__}: {exc}")
        raise



In [0]:
def main():
    # spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    # spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
    cfg = load_config(market_condition)
    required_cols = {
        "market",
        "source_database",
        "source_table",
        "target_path",
        "is_loading_blob_active",
    }
    missing = required_cols - set(cfg.columns)
    if missing:
        raise ValueError(f"Config missing columns: {', '.join(sorted(missing))}")

    if "primary_key" not in cfg.columns:
        cfg["primary_key"] = None
    if "pk_type" not in cfg.columns:
        cfg["pk_type"] = None

    cfg = cfg[cfg["is_loading_blob_active"] == True]

    items = cfg[
        [
            "market",
            "source_database",
            "source_table",
            "target_path",
            "primary_key",
            "pk_type",
        ]
    ].to_dict("records")

    # 表级并发：单表失败不阻断其它任务，末尾统一汇总失败数。
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_TABLES) as executor:
        futures = {
            executor.submit(process_table, item): item
            for item in items
        }
        failed = 0
        failed_tables = []
        failed_list = []
        for future in as_completed(futures):
            item = futures[future]
            trace_id = (
                f"[{item.get('market', '')}-{item.get('source_database', '')}-"
                f"{item.get('source_table', '')}]"
            )
            try:
                future.result()
            except Exception as exc:
                failed += 1
                failed_tables.append(trace_id)
                failed_list.append(f"{trace_id} [FAILED] {type(exc).__name__}: {exc}")

                print_log(f"{trace_id} [FAILED] {type(exc).__name__}: {exc}")
                print_log(f"{trace_id} [STACKTRACE]\n{traceback.format_exc()}")

        return failed, failed_list
        
# start loading
failed, failed_list = main()

In [0]:
if failed > 0:
    raise RuntimeError(f"Load completed with {failed} failed table tasks")

In [0]:
for i in failed_list:
    print("="*150)
    print(i)

In [0]:
# import pyspark.sql.functions as F

# optin_df, lower_bound, upper_bound = read_mysql_int_pk(
#     market ="HKG",
#     database= "hkg_elcconsumermdm",
#     table_name= "sconsumeroptin",
#     id_key = "scop_id"
# )

# display(optin_df.agg(
#     F.count("*"),
#     F.countDistinct("scop_id")
# ))

In [0]:
# d_optin_df = spark.sql(" select * from  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/HKG/hkg_elcconsumermdm/sconsumeroptin` ")